In [ ]:
# Library imports
import os, sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
sys.path.append(os.path.abspath('..'))

# File imports
from env import InventoryEnv
from miscellaneous import read_instance
from policies.perfect_hindsight_policy import PerfectHindsightPolicy
from policies.exact_value_function_policy import ExactValueFunctionPolicy
from policies.auxiliaries.imitation_learning_model import ImitationLearningNet

In [ ]:
# Function to run a single episode of the environment with the given policy and save the data to a CSV file

def run_episode(env, policy, instance_file, output_dir):
    
    num_warehouses = env.num_warehouses
    num_customers = env.num_customers
    capacity_distribution = env.capacity_distribution

    instance = read_instance(instance_file)
    env.set_instance(instance)
    policy.reset(instance)
    state, info = env.reset()

    done = False
    data = []

    while not done:

        action = policy.act(state)

        warehouses_distance = state["warehouses_distance"]
        warehouses_capacity = state["warehouses_capacity"]

        # Normalize the state features and append both the state and the action to the data list
        row = []

        for i in range(state['static_info']['num_warehouses']):
            row.append(warehouses_distance[i] / 212.13)
            row.append(warehouses_capacity[i] / state['static_info']['warehouses_initial_capacity'][i])

        row.append(action)
        data.append(row)

        state, reward, done, truncated, _ = env.step(action)

    #Save data in a csv file
    df = pd.DataFrame(data, columns=[f"warehouses_distance_{i}" for i in range(num_warehouses)] + [f"warehouses_capacity_{i}" for i in range(num_warehouses)] + ["action"])

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path / f"data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv", mode='a', header=False, index=False)

In [ ]:
#Optimal policy is used in our imitation learning training, but other policies could be used as well, depending on the behavior we want to imitate.

EXPERT_OUTPUT_DIRS = {
    "exact_value_function": "imitation_learning_training/perfect_agent_data"
}

def make_expert_policy(expert_name, env, num_warehouses):
    if expert_name == "exact_value_function":
        return ExactValueFunctionPolicy(env, num_warehouses, env.num_customers, env.capacity_distribution)

# Run 5000 episodes of the environment with the expert policy and save the data to CSV files for imitation learning training
# Returns the time taken to collect the data in minutes
def export_data(num_warehouses, num_customers, capacity_distribution, expert_name="exact_value_function"):

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    policy = make_expert_policy(expert_name, env, num_warehouses)

    num_instances = 5_000
    instances = [f'../instances/instances_train_imitation_learning/instances_seed_{i}.json' for i in range(20001,num_instances+20001)]

    output_dir = EXPERT_OUTPUT_DIRS[expert_name]

    output_path = Path(output_dir) / f"data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv"
    output_path.unlink(missing_ok=True)

    start_time = time.perf_counter()
    for instance in instances:
        run_episode(env, policy, instance, output_dir)
    data_collection_time_minutes = (time.perf_counter() - start_time) / 60

    return data_collection_time_minutes

In [ ]:
EXPERT_NAME = "exact_value_function"

num_customers = 50
data_collection_times = {}  # (num_warehouses, num_customers, capacity_distribution) -> minutes

for num_warehouses in [2, 3, 4, 5]:
    for capacity_distribution in ["uniform", "uneven"]:
        data_collection_time_minutes = export_data(num_warehouses, num_customers, capacity_distribution, expert_name=EXPERT_NAME)
        data_collection_times[(num_warehouses, num_customers, capacity_distribution)] = round(data_collection_time_minutes, 2)

In [ ]:
def nn_model(num_warehouses, num_customers, capacity_distribution, input_dir="imitation_learning_training/perfect_agent_data"):

    # Read the expert data from the CSV file
    df = pd.read_csv(
        f"{input_dir}/data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv",
        header=None
    )

    # Rename columns to alternate distance/capacity
    columns = []
    for i in range(num_warehouses):
        columns.append(f"d_{i}")
        columns.append(f"c_{i}")
    columns.append("action")
    df.columns = columns

    # Prepare the input and output data
    X = df.drop(columns=["action"]).values.astype(np.float32)   #state features
    y = df["action"].values.astype(np.int64)    #action

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Further split the training data into train/validation
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    X_train_t, y_train_t = torch.tensor(X_train), torch.tensor(y_train)
    X_val_t, y_val_t = torch.tensor(X_val), torch.tensor(y_val)
    X_test_t, y_test_t = torch.tensor(X_test), torch.tensor(y_test)

    # Reset the random seeds for reproducibility
    SEED = 42
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # Build the neural network model
    model = ImitationLearningNet(num_warehouses)
    optimizer = torch.optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss()

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),  # pins the per-epoch shuffle order too
    )

    # Train the model and after each epoch evaluate on the validation set, printing the loss and accuracy
    # While we do not use early stopping, we do print the validation loss and accuracy after each epoch to monitor the training process.
    epochs = 25
    start_time = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_accuracy = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()
        print(f"Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy:.4f}")

    # Get the total training time in minutes
    training_time_minutes = (time.perf_counter() - start_time) / 60

    # Evaluate the model
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        accuracy = (test_logits.argmax(dim=1) == y_test_t).float().mean().item()
    print(f"Accuracy: {accuracy}")

    # Save the model
    torch.save(
        model.state_dict(),
        f"imitation_learning_training/imitation_learning_models/model_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.pt"
    )

    return training_time_minutes, accuracy

In [ ]:
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ["uniform", "uneven"]
num_customers = 50  # Policy trained in small instances with 50 customers, then applied to larger instances (simulates the real-world scenario where the expert knowledge/optimal policy is only available on a small set of instances and then applied to larger instances)

accuracy_dict = {}
training_times = {}  # (num_warehouses, num_customers, capacity_distribution) -> minutes

for num_warehouses in num_warehouses_options:
    for capacity_distribution in capacity_distribution_options:
        training_time_minutes, accuracy = nn_model(num_warehouses, num_customers, capacity_distribution, input_dir=EXPERT_OUTPUT_DIRS[EXPERT_NAME])
        training_times[(num_warehouses, num_customers, capacity_distribution)] = round(training_time_minutes, 2)
        accuracy_dict[(num_warehouses, num_customers, capacity_distribution)] = accuracy

In [ ]:
# Stores the training times, data collection times, and accuracy in a CSV file for later analysis
rows = []
for key, data_collection_time in data_collection_times.items():
    num_warehouses, num_customers, capacity_distribution = key
    rows.append({
        'num_warehouses': num_warehouses,
        'num_customers': num_customers,
        'capacity_distribution': capacity_distribution,
        'data_collection_time': data_collection_time,
        'function_fitting_time': training_times[key],
        'training_time': training_times[key] + data_collection_time,
        'accuracy': accuracy_dict[key]
    })

info_dir = 'imitation_learning_training'
os.makedirs(info_dir, exist_ok=True)

training_times_df = pd.DataFrame(rows)
training_times_df.to_csv(os.path.join(info_dir, 'imitation_learning_training_times.csv'), index=False, float_format='%.5f')